In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pyspark.sql.functions as F
catalog_name='website'

In [0]:
gold_df=spark.read.table(catalog_name+'.silver.slv_metadata')

In [0]:
gold_df.count()
gold_df.printSchema()

### dim_user

In [0]:
dim_user=(
    gold_df
    .select('user_id')
    .filter(col('user_id').isNotNull())
    .distinct()
    .orderBy('user_id')
)
display(dim_user)
dim_user.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.gold.dim_user')

### dim_page

In [0]:
dim_page=(
    gold_df
    .select('page_url')
    .filter(col('page_url').isNotNull())
    .distinct()
    .withColumn('page_id',monotonically_increasing_id())
    .select('page_id','page_url')
)
display(dim_page)
display(dim_page.printSchema())
dim_page.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.gold.dim_page')

### dim_device

In [0]:
dim_device=(
    gold_df
    .select('browser','os','device_type','language')
    .distinct()
    .withColumn('device_id',monotonically_increasing_id())
    .select('device_id','browser','os','device_type','language')
)
display(dim_device)
display(dim_device.printSchema())
dim_device.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.gold.dim_device')

### dim_skill

In [0]:
dim_skill=(
    gold_df.select('skill_name','skill_type','skill_level')
    .filter(col('skill_name').isNotNull())
    .distinct()
    .withColumn('skill_id',monotonically_increasing_id())
    .select('skill_id','skill_name','skill_type','skill_level')
)
display(dim_skill)
display(dim_skill.printSchema())
dim_skill.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.gold.dim_skill')

### dim_date

In [0]:
dim_date=(
    gold_df
    .filter(col('timestamp').isNotNull())
    .select(col('timestamp').alias('full_timestamp'))
    .distinct()
    .withColumn('time_id',monotonically_increasing_id())
    .withColumn('date',to_date('full_timestamp'))
    .withColumn('year',year('full_timestamp'))
    .withColumn('quarter',quarter('full_timestamp'))
    .withColumn('month',month('full_timestamp'))
    .withColumn('day',dayofmonth('full_timestamp'))
    .withColumn('weekday',date_format('full_timestamp','EEEE'))
    .withColumn('hour',hour('full_timestamp'))
    .withColumn('minute',minute('full_timestamp'))
    .select('time_id','full_timestamp','date','year','quarter','month',
            'day','weekday','hour','minute')
)
display(dim_date.limit(15))
display(dim_date.printSchema())
dim_date.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.gold.dim_date')

time_id,full_timestamp,date,year,quarter,month,day,weekday,hour,minute
0,2026-07-03T13:11:52.094Z,2026-07-03,2026,3,7,3,Friday,13,11
1,2026-07-18T20:29:29.258Z,2026-07-18,2026,3,7,18,Saturday,20,29
2,2026-07-18T20:27:05.302Z,2026-07-18,2026,3,7,18,Saturday,20,27
3,2026-07-19T18:58:23.738Z,2026-07-19,2026,3,7,19,Sunday,18,58
4,2026-07-03T18:11:05.390Z,2026-07-03,2026,3,7,3,Friday,18,11
5,2026-07-19T21:48:26.428Z,2026-07-19,2026,3,7,19,Sunday,21,48
6,2026-07-19T18:41:17.778Z,2026-07-19,2026,3,7,19,Sunday,18,41
7,2026-07-19T21:09:04.322Z,2026-07-19,2026,3,7,19,Sunday,21,9
8,2026-07-20T08:40:49.195Z,2026-07-20,2026,3,7,20,Monday,8,40
9,2026-07-23T18:18:08.482Z,2026-07-23,2026,3,7,23,Thursday,18,18


root
 |-- time_id: long (nullable = false)
 |-- full_timestamp: timestamp (nullable = true)
 |-- date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- weekday: string (nullable = true)
 |-- hour: integer (nullable = true)
 |-- minute: integer (nullable = true)



### fact_events

In [0]:
fact_events=(
    gold_df.alias('g')
    .join(dim_user.alias('u'),col('g.user_id')==col('u.user_id'),'left')
    .join(dim_page.alias('p'),col('g.page_url')==col('p.page_url'),'left')
    .join(dim_device.alias('d'),(col('g.browser')==col('d.browser'))&
        (col('g.browser')==col('d.browser'))&
        (col('g.os')==col('d.os'))&
        (col('g.device_type')==col('d.device_type'))&
        (col('g.language')==col('d.language')),'left')
    .join(dim_skill.alias('s'),
          (col('g.skill_name').eqNullSafe(col('s.skill_name')))&
          (col('g.skill_type').eqNullSafe(col('s.skill_type')))&
          (col('g.skill_level').eqNullSafe(col('s.skill_level'))),'left')
    .join(dim_date.alias('dt'),col('g.timestamp')==col('dt.full_timestamp'),'left')
    .select(
        col('g.id').alias('event_id'),
        col('u.user_id'),
        col('p.page_id'),
        col('d.device_id'),
        col('s.skill_id'),
        col('dt.time_id'),
        col('g.session_id'),
        col('g.event_type'),
        col('g.score'),
        col('g.duration').alias('duration_ms'),
        col('g.referrer')
    )                                  
)
display(fact_events.limit(30))
display(fact_events.printSchema())
fact_events.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.gold.fact_events')

event_id,user_id,page_id,device_id,skill_id,time_id,session_id,event_type,score,duration_ms,referrer
1,null,12,0,null,292,sess_yiy30vljdw_1783042842796,page_view,null,null,https://vercel.com/
2,null,1,0,null,687,sess_yiy30vljdw_1783042842796,page_view,null,null,https://vercel.com/
3,1,12,0,null,96,sess_yiy30vljdw_1783042842796,page_view,null,null,https://vercel.com/
4,1,6,0,null,391,sess_yiy30vljdw_1783042842796,page_view,null,null,https://vercel.com/
5,1,6,0,25,42,sess_yiy30vljdw_1783042842796,quiz_completed,100.0,null,https://vercel.com/
6,1,6,0,11,223,sess_yiy30vljdw_1783042842796,skill_added,null,null,https://vercel.com/
7,1,6,0,23,109,sess_yiy30vljdw_1783042842796,skill_added,null,null,https://vercel.com/
8,1,2,0,null,580,sess_yiy30vljdw_1783042842796,page_view,null,null,https://vercel.com/
9,1,15,0,null,290,sess_yiy30vljdw_1783042842796,page_view,null,null,https://vercel.com/
10,1,12,0,null,680,sess_yiy30vljdw_1783042842796,page_view,null,null,https://vercel.com/


root
 |-- event_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- page_id: long (nullable = true)
 |-- device_id: long (nullable = true)
 |-- skill_id: long (nullable = true)
 |-- time_id: long (nullable = true)
 |-- session_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- score: double (nullable = true)
 |-- duration_ms: long (nullable = true)
 |-- referrer: string (nullable = true)



### 1. Row count validation

In [0]:
assert fact_events.count() == gold_df.count(), \
    "Fact row count does not match Gold row count"

### 2. Primary Key validation

In [0]:
assert fact_events.select("event_id").distinct().count() == fact_events.count(), \
    "Duplicate event_id found"

### 3. Foreign Key validation

In [0]:
for col_name in ["user_id", "page_id", "device_id", "skill_id", "time_id"]:
    null_count = fact_events.filter(col(col_name).isNull()).count()
    print(f"{col_name}: {null_count} NULLs")

user_id: 170 NULLs
page_id: 0 NULLs
device_id: 0 NULLs
skill_id: 679 NULLs
time_id: 0 NULLs


### 4. Business/Data Quality validation

In [0]:
assert fact_events.filter(col("duration_ms") < 0).count() == 0, \
    "Negative duration found"

assert fact_events.filter(col("score") < 0).count() == 0, \
    "Negative score found"